# Xem nhanh dữ liệu trong Postgres — table browser

Notebook này **chỉ đọc** (không ghi/sửa gì) — mục đích duy nhất là cho bạn xem nhanh nội
dung thật của từng bảng trong Postgres, sau đợt sửa lỗi migration `pdf_document_id`
(2026-07-30/31, xem `docs/PROGRESS.md` phần cuối để biết bối cảnh đầy đủ: 6 lỗi thật đã
sửa, kết thúc bằng `overall ok=True` trong `data_quality_report.py`).

Khác với `notebooks/01_data_inventory_eda.ipynb` (phân tích/EDA sâu, mục đích quyết định
kiến trúc ML) — notebook này đơn giản hơn nhiều: liệt kê bảng, đếm dòng, xem mẫu dữ liệu.
Không vẽ biểu đồ, không phân tích thống kê.

Chạy trực tiếp trên máy đã có:
- Docker Postgres đang chạy (`docker compose up -d`)
- `.venv` của project đã cài đủ dependency
- Kernel Jupyter trỏ vào `.venv/bin/python` của project

Mỗi cell được bọc `try/except` — một bảng lỗi/rỗng không làm vỡ cả notebook.

In [1]:
import sys
import traceback
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), "Chay notebook nay tu thu muc goc project (co folder src/)"
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

from src.core.db import engine, check_connection

check_connection()

OK - connected to localhost:5432/financial_db


True

## 1. Danh sách toàn bộ bảng + số dòng

Lấy động từ `information_schema` - tự cập nhật khi có bảng mới, không cần sửa tay notebook
này mỗi khi schema đổi.

In [2]:
try:
    table_names = pd.read_sql(
        """SELECT table_name FROM information_schema.tables
           WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
           ORDER BY table_name""", engine
    )["table_name"].tolist()

    counts = []
    for t in table_names:
        try:
            n = pd.read_sql(f"SELECT count(*) AS n FROM {t}", engine)["n"].iloc[0]
        except Exception as e:
            n = f"loi: {type(e).__name__}"
        counts.append({"table": t, "n_rows": n})

    overview = pd.DataFrame(counts)
    display(overview)
except Exception as e:
    print(f"[canh bao] khong liet ke duoc bang: {type(e).__name__}: {e}")
    traceback.print_exc()
    table_names = []

,table,n_rows
0,causal_experiment_runs,0
1,causal_panel_snapshot,0
2,companies,222
3,company_price_snapshot,39575
4,disclosure_chunks,661657
5,disclosures,3155
6,financial_line_items,283732
7,financial_ratios,136369
8,ingestion_log,432
9,macro_indicators,264


## 2. `pdf_documents` — bảng vừa sửa xong đợt migration này

Chú ý đặc biệt: cột `pdf_hash` — dòng nào bằng `MISSING_SOURCE_PDF_HASH_SENTINEL`
(64 số 0) nghĩa là file PDF gốc đã xác nhận không còn, đăng ký qua
`--allow-missing-pdf-placeholder` (xem `scripts/register_pdf_documents.py`).

In [3]:
try:
    from scripts.register_pdf_documents import MISSING_SOURCE_PDF_HASH_SENTINEL

    pdf_docs = pd.read_sql("SELECT * FROM pdf_documents ORDER BY id", engine)
    print(f"pdf_documents: {len(pdf_docs):,} dong")

    n_placeholder = (pdf_docs["pdf_hash"] == MISSING_SOURCE_PDF_HASH_SENTINEL).sum()
    print(f"  - {n_placeholder:,} dong dung placeholder hash (PDF goc da mat, xem cot pdf_hash)")
    print(f"  - {(~pdf_docs['is_active']).sum():,} dong is_active=FALSE (PDF da bi xoa khoi --input)")

    print("\nMau 20 dong dau:")
    display(pdf_docs.head(20))

    print("\n20 dong dung placeholder hash (neu co) - nhung file KHONG con PDF goc:")
    display(pdf_docs[pdf_docs["pdf_hash"] == MISSING_SOURCE_PDF_HASH_SENTINEL].head(20))
except Exception as e:
    print(f"[canh bao] khong doc duoc pdf_documents: {type(e).__name__}: {e}")
    traceback.print_exc()
    pdf_docs = pd.DataFrame()

pdf_documents: 3,572 dong
  - 2,890 dong dung placeholder hash (PDF goc da mat, xem cot pdf_hash)
  - 0 dong is_active=FALSE (PDF da bi xoa khoi --input)

Mau 20 dong dau:


,id,source_file,pdf_hash,ticker,doctype,fiscal_year,n_pages_total,extraction_version,extracted_at,document_version,is_active,linked_disclosure_id,updated_at
0,2,01.-Thong-bao-moi-hop-DHDCD-2023_VN-Final.pdf,7243c1bdf983c43976d2915b76b835d2b902af79d475e07accfa6ff12af79922,UNKNOWN,other,2023.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
1,3,01.-Thong-bao-moi-hop-DHDCD-2024_VN-Final.pdf,d6736c805129c97eec9921c8587313c5effc5cf032c7abe68669d1fa175377a8,UNKNOWN,other,2024.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
2,4,04.-HDSD-Bvote-cho-CD-VN.pdf,065c8a577a8ff57cf4db37a25e518cf6477893bbbd904c75a21d4b3080766df5,UNKNOWN,other,NaN,6,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
3,5,0_Ban_CBTT_Tai_lieu_AGM_2026_f295e68b32.pdf,63fa42bdbe4a098eddf6eccfe025b8747ca303da8b39a6f78ab43b1f06188177,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
4,6,10_AGM_2026_Quy_che_to_chuc_dai_hoi_5765b92317.pdf,b20fb9a9051e3179c29c439ad71a224409aae56d1e28ca6149d452a445e92349,UNKNOWN,other,2026.0,17,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
5,7,10_aah_2024_5_4_bbab271_12__to_trinh_kqsxkd_signed.pdf,3fd829beeaaaab9f508c9e21d28ef1146b55664083db5b646f9b187b673d654b,AAH,other,2024.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,38.0,2026-07-30 07:36:06.838922+00:00
6,8,11_AGM_2026_Notice_of_candidate_nomination_59bf87a429.pdf,1b81a86045b84d4f7e0b2ab4e2d6aa562bfcf994bda8c7ee894465029e0d96d1,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
7,9,11_AGM_2026_Thong_bao_de_cu_ung_cu_HDQT_VN_f5763c4c27.pdf,667a6c9d407dac2945f1bbdc6e58eae855dbfa01564edc239c3b3f8700973860,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
8,10,11_aah_2024_5_4_c3757c1_08__bc_ban_tgd_signed.pdf,3ffa767987072af0fd8297c3e2ea36931fd05a3ca387519a077676d7d82f1749,AAH,other,2024.0,4,,2026-07-30 07:36:06.838922+00:00,1,True,38.0,2026-07-30 07:36:06.838922+00:00
9,11,12_VNM_AGM_2026_Mau_1_A_Thu_de_cu_cua_co_dong_VN_Co_dong_don_le_10b9d06fd2.pdf,e8def24080e3e56c136d6d0e7bd607bb7a2baa6a156a9f55b5eaf6913d70ab1c,VNM,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00



20 dong dung placeholder hash (neu co) - nhung file KHONG con PDF goc:


,id,source_file,pdf_hash,ticker,doctype,fiscal_year,n_pages_total,extraction_version,extracted_at,document_version,is_active,linked_disclosure_id,updated_at
682,1707,1_aas_2022_3_31_503a68a_bctlattc_2021__cks,0000000000000000000000000000000000000000000000000000000000000000,AAS,other,2021.0,28,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00
683,1708,1_aav_2025_3_31_1029b13_en_consolidatedauditedfinancialstatements_2024,0000000000000000000000000000000000000000000000000000000000000000,AAV,other,2024.0,44,,2026-07-30 09:39:33.456660+00:00,1,True,95.0,2026-07-30 09:39:33.456660+00:00
684,1709,1_aav_2025_3_31_958df1a_en_separateauditedfinancialstatements_2024,0000000000000000000000000000000000000000000000000000000000000000,AAV,other,2024.0,41,,2026-07-30 09:39:33.456660+00:00,1,True,95.0,2026-07-30 09:39:33.456660+00:00
685,1710,1_abc_2025_3_31_4be07a6_financial__statement_2024_audited,0000000000000000000000000000000000000000000000000000000000000000,ABC,other,2024.0,45,,2026-07-30 09:39:33.456660+00:00,1,True,397.0,2026-07-30 09:39:33.456660+00:00
686,1711,1_abc_2026_3_26_28649d6_vi_baocaotaichinhkiemtoan2025,0000000000000000000000000000000000000000000000000000000000000000,ABC,financial_statement,2025.0,46,,2026-07-30 09:39:33.456660+00:00,1,True,404.0,2026-07-30 09:39:33.456660+00:00
687,1712,1_acg_2022_3_28_2703ca7_acg_bctc_hop_nhat_2021_signed,0000000000000000000000000000000000000000000000000000000000000000,ACG,other,2021.0,47,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00
688,1713,1.ACV_2020.4.14_b666973_BCTC_hop_nhat_2019da_nen,0000000000000000000000000000000000000000000000000000000000000000,ACV,other,2019.0,64,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00
689,1714,1.ACV_2021.4.12_1f69703_ACVBCTC_Hop_nhat_nam_2020_da_duoc_kiem_toancompressed,0000000000000000000000000000000000000000000000000000000000000000,ACV,other,2020.0,62,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00
690,1715,1_acv_2022_4_5_5f2b45d_acv__bao_cao_tai_chinh_hop_nhat,0000000000000000000000000000000000000000000000000000000000000000,ACV,financial_statement,2022.0,62,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00
691,1716,1_acv_2023_3_31_0884ea6_acv__bao_cao_tai_chinh_hop_nhat_2022,0000000000000000000000000000000000000000000000000000000000000000,ACV,financial_statement,2022.0,65,,2026-07-30 09:39:33.456660+00:00,1,True,NaN,2026-07-30 09:39:33.456660+00:00


## 3. `disclosure_chunks` — corpus RAG (mẫu, KHÔNG load hết 661k+ dòng vào RAM)

Bảng này rất lớn — luôn dùng `LIMIT` hoặc lọc theo `ticker` khi xem, đừng `SELECT *` không
giới hạn.

In [4]:
try:
    chunks_sample = pd.read_sql(
        """SELECT id, ticker, doctype, source_file, page, chunk_index, char_count,
                  pdf_document_id, (embedding IS NOT NULL) AS has_embedding,
                  (chunk_hash IS NOT NULL) AS has_chunk_hash
           FROM disclosure_chunks ORDER BY id LIMIT 20""", engine
    )
    print("Mau 20 dong dau (khong lay chunk_text/embedding day du - qua nang):")
    display(chunks_sample)

    total_chunks = pd.read_sql("SELECT count(*) AS n FROM disclosure_chunks", engine)["n"].iloc[0]
    print(f"\nTong so chunk: {total_chunks:,}")
except Exception as e:
    print(f"[canh bao] khong doc duoc disclosure_chunks: {type(e).__name__}: {e}")
    traceback.print_exc()

Mau 20 dong dau (khong lay chunk_text/embedding day du - qua nang):


,id,ticker,doctype,source_file,page,chunk_index,char_count,pdf_document_id,has_embedding,has_chunk_hash
0,1,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,1,0,273,2371,True,True
1,2,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,10,0,750,2371,True,True
2,3,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,10,1,782,2371,True,True
3,4,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,10,2,796,2371,True,True
4,5,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,10,3,791,2371,True,True
5,6,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,11,0,721,2371,True,True
6,7,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,11,1,718,2371,True,True
7,8,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,11,2,345,2371,True,True
8,9,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,11,3,120,2371,True,True
9,10,A32,financial_statement,A32_Baocaotaichinh_2017_Kiemtoan,11,4,119,2371,True,True



Tong so chunk: 661,657


In [5]:
# Xem chunk theo 1 ticker cu the - sua TICKER ben duoi
TICKER = "VNM"

try:
    by_ticker = pd.read_sql(
        """SELECT id, ticker, doctype, source_file, page, chunk_index, char_count
           FROM disclosure_chunks WHERE ticker = %(ticker)s ORDER BY source_file, page, chunk_index LIMIT 50""",
        engine, params={"ticker": TICKER}
    )
    print(f"disclosure_chunks WHERE ticker = '{TICKER}': {len(by_ticker)} dong (hien 50 dau)")
    display(by_ticker)
except Exception as e:
    print(f"[canh bao] khong query duoc theo ticker: {type(e).__name__}: {e}")
    traceback.print_exc()

disclosure_chunks WHERE ticker = 'VNM': 50 dong (hien 50 dau)


,id,ticker,doctype,source_file,page,chunk_index,char_count
0,1378234,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,1,0,131
1,1378272,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,2,0,334
2,1378314,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,3,0,798
3,1378315,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,3,1,797
4,1378316,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,3,2,794
5,1378357,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,4,0,796
6,1378358,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,4,1,797
7,1378359,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,4,2,754
8,1378401,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,5,0,797
9,1378402,VNM,other,BCTC HOP NHAT 31.12.2019 - VN,5,1,795


## 4. Dữ liệu cấu trúc (DAG path) — `companies` / `financial_line_items` / `financial_ratios` / `disclosures`

In [6]:
for table in ["companies", "financial_line_items", "financial_ratios", "disclosures"]:
    try:
        df = pd.read_sql(f"SELECT * FROM {table} LIMIT 20", engine)
        n_total = pd.read_sql(f"SELECT count(*) AS n FROM {table}", engine)["n"].iloc[0]
        print(f"--- {table}: {n_total:,} dong (hien 20 dau) ---")
        display(df)
    except Exception as e:
        print(f"[canh bao] khong doc duoc {table}: {type(e).__name__}: {e}")
        traceback.print_exc()

--- companies: 222 dong (hien 20 dau) ---


,ticker,company_name,exchange,sector,created_at
0,VNM,CTCP Sữa Việt Nam,HOSE,Tiêu dùng thiết yếu,2026-07-27 10:17:41.540256+00:00
1,SAB,Tổng Công ty cổ phần Bia - Rượu - Nước giải khát Sài Gòn,HOSE,Tiêu dùng thiết yếu,2026-07-27 10:17:41.540256+00:00
2,MSN,CTCP Tập đoàn Masan,HOSE,Tiêu dùng thiết yếu,2026-07-27 10:17:41.540256+00:00
3,KDC,CTCP Tập đoàn KIDO,HOSE,Tiêu dùng thiết yếu,2026-07-27 10:17:41.540256+00:00
4,VIC,Tập đoàn VINGROUP - CTCP,HOSE,Bất động sản,2026-07-27 10:17:41.540256+00:00
5,VHM,CTCP Vinhomes,HOSE,Bất động sản,2026-07-27 10:17:41.540256+00:00
6,NVL,CTCP Tập đoàn Đầu tư Địa ốc No Va,HOSE,Bất động sản,2026-07-27 10:17:41.540256+00:00
7,KDH,CTCP Đầu tư và Kinh doanh Nhà Khang Điền,HOSE,Bất động sản,2026-07-27 10:17:41.540256+00:00
8,HPG,CTCP Tập đoàn Hòa Phát,HOSE,Công nghiệp chế biến,2026-07-27 10:17:41.540256+00:00
9,GAS,Tổng Công ty Khí Việt Nam - CTCP,HOSE,Công nghiệp chế biến,2026-07-27 10:17:41.540256+00:00


--- financial_line_items: 283,732 dong (hien 20 dau) ---


,id,ticker,fiscal_year,period,ms_code,item_name,statement_type,consolidation,value,unit,updated_at
0,91047,API,2024,Q1,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.736632e+11,VND,2026-07-27 10:17:56.259173+00:00
1,91048,API,2024,Q2,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.466103e+11,VND,2026-07-27 10:17:56.259173+00:00
2,91049,API,2024,Q3,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.338423e+11,VND,2026-07-27 10:17:56.259173+00:00
3,91050,API,2024,Q4,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.354500e+11,VND,2026-07-27 10:17:56.259173+00:00
4,91051,API,2024,H1,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.480193e+11,VND,2026-07-27 10:17:56.259173+00:00
5,91052,API,2024,9M,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.338423e+11,VND,2026-07-27 10:17:56.259173+00:00
6,91053,API,2025,YEAR,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.044742e+11,VND,2026-07-27 10:17:56.259173+00:00
7,91054,API,2025,Q1,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.140540e+11,VND,2026-07-27 10:17:56.259173+00:00
8,91055,API,2025,Q2,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.089465e+11,VND,2026-07-27 10:17:56.259173+00:00
9,91056,API,2025,Q3,140,IV. Hàng tồn kho,balance_sheet,consolidated,8.082158e+11,VND,2026-07-27 10:17:56.259173+00:00


--- financial_ratios: 136,369 dong (hien 20 dau) ---


,id,ticker,fiscal_year,period,ratio_group,ratio_name,value,unit,updated_at
0,1,A32,2024,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS),5775.00,VNĐ,2026-07-27 10:18:23.288957+00:00
1,2,A32,2025,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Thu nhập trên mỗi cổ phần của 4 quý gần nhất (EPS),7481.00,VNĐ,2026-07-27 10:18:23.288957+00:00
2,3,A32,2024,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Giá trị sổ sách của cổ phiếu (BVPS),32525.00,VNĐ,2026-07-27 10:18:23.288957+00:00
3,4,A32,2025,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Giá trị sổ sách của cổ phiếu (BVPS),33937.00,VNĐ,2026-07-27 10:18:23.288957+00:00
4,5,A32,2024,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên thu nhập (P/E),6.61,Lần,2026-07-27 10:18:23.288957+00:00
5,6,A32,2025,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên thu nhập (P/E),4.67,Lần,2026-07-27 10:18:23.288957+00:00
6,7,A32,2024,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên giá trị sổ sách (P/B),1.17,Lần,2026-07-27 10:18:23.288957+00:00
7,8,A32,2025,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên giá trị sổ sách (P/B),1.03,Lần,2026-07-27 10:18:23.288957+00:00
8,9,A32,2024,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên doanh thu thuần (P/S),0.36,Lần,2026-07-27 10:18:23.288957+00:00
9,10,A32,2025,YEAR,NHÓM CHỈ SỐ ĐỊNH GIÁ,Chỉ số giá thị trường trên doanh thu thuần (P/S),0.31,Lần,2026-07-27 10:18:23.288957+00:00


--- disclosures: 3,155 dong (hien 20 dau) ---


,id,ticker,fiscal_year,period,consolidation,report_type,audit_status,auditor,audit_opinion,disclosure_date,signed_date,source_pdf_path,structure_valid,updated_at
0,1,A32,2024,YEAR,standalone,nam,Kiểm toán,VAE,Chấp nhận toàn phần,2025-03-04,2025-02-26,None,None,2026-07-27 10:17:44.859436+00:00
1,2,A32,2025,YEAR,standalone,nam,Kiểm toán,VAE,Chấp nhận toàn phần,2026-04-01,2026-03-28,None,None,2026-07-27 10:17:44.859436+00:00
2,1842,BTP,2024,YEAR,standalone,nam,Kiểm toán,Deloitte,Chấp nhận toàn phần,2025-03-27,2025-03-25,None,None,2026-07-27 10:18:06.444263+00:00
3,1843,BTP,2024,Q1,standalone,quy,Chưa kiểm toán,None,None,2024-04-16,None,None,None,2026-07-27 10:18:06.444263+00:00
4,1844,BTP,2024,Q2,standalone,quy,Chưa kiểm toán,None,None,2024-07-17,None,None,None,2026-07-27 10:18:06.444263+00:00
5,1845,BTP,2024,Q3,standalone,quy,Chưa kiểm toán,None,None,2024-10-18,None,None,None,2026-07-27 10:18:06.444263+00:00
6,1846,BTP,2024,Q4,standalone,quy,Chưa kiểm toán,None,None,2025-01-21,None,None,None,2026-07-27 10:18:06.444263+00:00
7,1847,BTP,2024,H1,standalone,ban_nien,Soát xét,Deloitte,Chấp nhận toàn phần,2024-08-15,2024-08-14,None,None,2026-07-27 10:18:06.444263+00:00
8,1848,BTP,2024,9M,standalone,ban_nien,Chưa kiểm toán,None,None,2024-10-18,None,None,None,2026-07-27 10:18:06.444263+00:00
9,1849,BTP,2025,YEAR,standalone,nam,Kiểm toán,Deloitte,Chấp nhận toàn phần,2026-04-01,2026-03-31,None,None,2026-07-27 10:18:06.444263+00:00


## 5. `predictive_features` / `predictive_split_manifest` / `causal_panel_snapshot` / `causal_experiment_runs`

Các bảng phục vụ nhánh PREDICTIVE/CAUSAL (nếu chưa chạy các script tương ứng, bảng sẽ rỗng
- không phải lỗi).

In [7]:
for table in ["predictive_features", "predictive_split_manifest", "causal_panel_snapshot", "causal_experiment_runs"]:
    try:
        df = pd.read_sql(f"SELECT * FROM {table} LIMIT 20", engine)
        n_total = pd.read_sql(f"SELECT count(*) AS n FROM {table}", engine)["n"].iloc[0]
        print(f"--- {table}: {n_total:,} dong (hien 20 dau) ---")
        display(df)
    except Exception as e:
        print(f"[canh bao] khong doc duoc {table}: {type(e).__name__}: {e}")
        traceback.print_exc()

--- predictive_features: 0 dong (hien 20 dau) ---


,id,feature_version,ticker,fiscal_year,period,consolidation,sector,exchange,total_assets,total_liabilities,total_equity,current_assets,current_liabilities,inventory,revenue,cogs,gross_profit,profit_before_tax,net_profit,operating_cash_flow,leverage,current_ratio,gross_margin,net_margin,asset_turnover,ocf_to_revenue,total_assets_lag1,total_liabilities_lag1,total_equity_lag1,current_assets_lag1,...,gross_profit_lag1,profit_before_tax_lag1,net_profit_lag1,operating_cash_flow_lag1,leverage_lag1,current_ratio_lag1,gross_margin_lag1,net_margin_lag1,asset_turnover_lag1,ocf_to_revenue_lag1,revenue_growth_yoy,net_profit_growth_yoy,total_assets_growth_yoy,cpi_annual_avg,gdp_growth_annual_avg,gold_price_annual_avg,usdvnd_annual_avg,oil_price_annual_avg,information_cutoff,target_col,horizon,target_value,eligible_for_training,created_at,updated_at,target_observed_at,target_release_date,feature_created_at,feature_hash,source_snapshot_version


--- predictive_split_manifest: 0 dong (hien 20 dau) ---


,id,split_manifest_version,feature_version,ticker,fiscal_year,period,consolidation,target_col,horizon,information_cutoff,split,frozen_at


--- causal_panel_snapshot: 0 dong (hien 20 dau) ---


,id,snapshot_version,ticker,fiscal_year,period,consolidation,sector,exchange,total_assets,total_liabilities,total_equity,current_assets,current_liabilities,inventory,revenue,cogs,gross_profit,profit_before_tax,net_profit,operating_cash_flow,leverage,current_ratio,gross_margin,net_margin,asset_turnover,ocf_to_revenue,cpi_annual_avg,gdp_growth_annual_avg,gold_price_annual_avg,usdvnd_annual_avg,oil_price_annual_avg,eligible_sample,exclusion_reason,created_at


--- causal_experiment_runs: 0 dong (hien 20 dau) ---


,id,snapshot_version,source_dataset_version,treatment_name,outcome_name,covariates,estimator,nuisance_model,random_seed,bootstrap_iterations,min_sample_threshold,code_commit,n,n_treated,n_control,point_estimate,ci_low,ci_high,confidence_label,ok,error_message,created_at


## 6. `macro_indicators` / `company_price_snapshot` (market-factor & giá)

In [8]:
for table in ["macro_indicators", "company_price_snapshot"]:
    try:
        df = pd.read_sql(f"SELECT * FROM {table} LIMIT 20", engine)
        n_total = pd.read_sql(f"SELECT count(*) AS n FROM {table}", engine)["n"].iloc[0]
        print(f"--- {table}: {n_total:,} dong (hien 20 dau) ---")
        display(df)
    except Exception as e:
        print(f"[canh bao] khong doc duoc {table}: {type(e).__name__}: {e}")
        traceback.print_exc()

--- macro_indicators: 264 dong (hien 20 dau) ---


,id,category,indicator_name,unit,period_type,fiscal_year,period,value,source_file,updated_at,release_date
0,1,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0728,1557.42,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
1,2,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0729,1493.41,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
2,3,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0730,1507.63,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
3,4,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0731,1502.52,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
4,5,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0801,1495.21,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
5,6,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0804,1528.19,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
6,7,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0805,1547.15,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
7,8,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0806,1573.71,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
8,9,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0807,1581.81,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None
9,10,MKT_VNINDEX,VN-Index (close),điểm,DAILY,2025,D0808,1584.95,vnstock_vci,2026-07-27 10:17:47.576479+00:00,None


--- company_price_snapshot: 39,575 dong (hien 20 dau) ---


,id,ticker,trade_date,open_price,high_price,low_price,close_price,volume,source,updated_at
0,1,A32,2025-07-30,32.54,32.64,32.54,32.64,500,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
1,2,A32,2025-07-31,32.82,32.82,32.82,32.82,100,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
2,3,A32,2025-08-01,32.73,33.01,32.73,33.01,400,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
3,4,A32,2025-08-04,32.73,32.73,32.73,32.73,200,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
4,5,A32,2025-08-05,32.73,32.73,32.73,32.73,200,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
5,6,A32,2025-08-06,33.01,36.54,33.01,36.54,200,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
6,7,A32,2025-08-07,33.20,33.20,33.20,33.20,100,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
7,8,A32,2025-08-08,33.20,33.20,33.20,33.20,100,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
8,9,A32,2025-08-11,34.31,34.31,34.31,34.31,100,vnstock_kbs,2026-07-27 10:17:48.222030+00:00
9,10,A32,2025-08-12,34.31,34.31,34.31,34.31,100,vnstock_kbs,2026-07-27 10:17:48.222030+00:00


## 7. Xem bảng bất kỳ (tự sửa tên bảng)

Đổi `TABLE_NAME` bên dưới thành tên bất kỳ trong danh sách ở mục 1, chạy lại cell.

In [9]:
TABLE_NAME = "pdf_documents"  # doi ten bang o day

try:
    df = pd.read_sql(f"SELECT * FROM {TABLE_NAME} LIMIT 50", engine)
    print(f"{TABLE_NAME}: hien {len(df)} dong dau")
    display(df)
except Exception as e:
    print(f"[canh bao] khong doc duoc {TABLE_NAME}: {type(e).__name__}: {e}")
    traceback.print_exc()

pdf_documents: hien 50 dong dau


,id,source_file,pdf_hash,ticker,doctype,fiscal_year,n_pages_total,extraction_version,extracted_at,document_version,is_active,linked_disclosure_id,updated_at
0,2,01.-Thong-bao-moi-hop-DHDCD-2023_VN-Final.pdf,7243c1bdf983c43976d2915b76b835d2b902af79d475e07accfa6ff12af79922,UNKNOWN,other,2023.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
1,3,01.-Thong-bao-moi-hop-DHDCD-2024_VN-Final.pdf,d6736c805129c97eec9921c8587313c5effc5cf032c7abe68669d1fa175377a8,UNKNOWN,other,2024.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
2,4,04.-HDSD-Bvote-cho-CD-VN.pdf,065c8a577a8ff57cf4db37a25e518cf6477893bbbd904c75a21d4b3080766df5,UNKNOWN,other,NaN,6,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
3,5,0_Ban_CBTT_Tai_lieu_AGM_2026_f295e68b32.pdf,63fa42bdbe4a098eddf6eccfe025b8747ca303da8b39a6f78ab43b1f06188177,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
4,6,10_AGM_2026_Quy_che_to_chuc_dai_hoi_5765b92317.pdf,b20fb9a9051e3179c29c439ad71a224409aae56d1e28ca6149d452a445e92349,UNKNOWN,other,2026.0,17,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
5,343,01.-Thong-bao-moi-hop-DHDCD-2023_VN-Final,7243c1bdf983c43976d2915b76b835d2b902af79d475e07accfa6ff12af79922,UNKNOWN,other,2023.0,1,,2026-07-30 08:43:37.991717+00:00,1,True,NaN,2026-07-30 08:43:37.991717+00:00
6,7,10_aah_2024_5_4_bbab271_12__to_trinh_kqsxkd_signed.pdf,3fd829beeaaaab9f508c9e21d28ef1146b55664083db5b646f9b187b673d654b,AAH,other,2024.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,38.0,2026-07-30 07:36:06.838922+00:00
7,8,11_AGM_2026_Notice_of_candidate_nomination_59bf87a429.pdf,1b81a86045b84d4f7e0b2ab4e2d6aa562bfcf994bda8c7ee894465029e0d96d1,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
8,9,11_AGM_2026_Thong_bao_de_cu_ung_cu_HDQT_VN_f5763c4c27.pdf,667a6c9d407dac2945f1bbdc6e58eae855dbfa01564edc239c3b3f8700973860,UNKNOWN,other,2026.0,1,,2026-07-30 07:36:06.838922+00:00,1,True,NaN,2026-07-30 07:36:06.838922+00:00
9,344,01.-Thong-bao-moi-hop-DHDCD-2024_VN-Final,d6736c805129c97eec9921c8587313c5effc5cf032c7abe68669d1fa175377a8,UNKNOWN,other,2024.0,1,,2026-07-30 08:43:37.991717+00:00,1,True,NaN,2026-07-30 08:43:37.991717+00:00
